In [21]:
import pandas as pd
import json
import ast

# 23/24


In [22]:
# load fgo json data
try:
    # 2. Open the file using the 'with' statement
    with open('./fantasyGO_data.json', 'r') as f:
        # 3. Load the JSON data from the file
        fgo_23_24 = json.load(f)

    # The 'data' variable now holds your JSON content
    # In Jupyter, placing the variable at the end of the cell will display it
    print("✅ JSON file loaded successfully!")

except FileNotFoundError:
    print(f"❌ Error: The file ./fantasyGO_data.json was not found. Please check the path.")
except json.JSONDecodeError:
    print(f"❌ Error: The file ./fantasyGO_data.json is not a valid JSON file. Please check its contents.")

✅ JSON file loaded successfully!


In [23]:
fgo_picks = []

# Loop through each league in the top-level list
for gameweek_data in fgo_23_24:
    gameweek_info = {
        'gameweek': gameweek_data.get('contest').split(' ')[1],
        'contestant_no': gameweek_data.get('contestant_no'),
        'prize_pool': gameweek_data.get('prize_pool')
    }

    # Loop through each page within the league (e.g., "page_1", "page_2")
    for i in range(1, 1000): # Assuming a max of 999 pages
        page_key = f'page_{i}'
        if page_key in gameweek_data:
            # Loop through each manager's entry on the page
            for entry_data in gameweek_data[page_key]:
                entry_info = {
                    'manager': entry_data.get('manager'),
                    'entry_num': entry_data.get('entry'),
                    'total_points': entry_data.get('points'),
                    'prize': entry_data.get('prize')
                }

                # Loop through each player pick in the entry
                for pick_data in entry_data.get('pick', []):
                    # Combine all the info into a single record
                    full_record = {
                        **gameweek_info,
                        **entry_info,
                        **pick_data
                    }
                fgo_picks.append({**gameweek_info, **entry_data})
        else:
            break # Stop if the next page doesn't exist

pd.DataFrame(fgo_picks).to_csv('./fgo_managers.csv', index=False)

In [24]:
# 2. Flatten the nested fgo_23_24
all_picks = []

# Loop through each league in the top-level list
for gameweek_data in fgo_23_24:
    gameweek_info = {
        'gameweek': gameweek_data.get('contest').split(' ')[1],
        'contestant_no': gameweek_data.get('contestant_no'),
        'prize_pool': gameweek_data.get('prize_pool')
    }
    # Loop through each page within the league (e.g., "page_1", "page_2")
    for i in range(1, 100): # Assuming a max of 99 pages
        page_key = f'page_{i}'
        if page_key in gameweek_data:
            # Loop through each manager's entry on the page
            for entry_data in gameweek_data[page_key]:
                entry_info = {
                    'manager': entry_data.get('manager'),
                    'entry_num': entry_data.get('entry'),
                    'total_points': entry_data.get('points'),
                    'prize': entry_data.get('prize')
                }

                # Loop through each player pick in the entry
                for pick_data in entry_data.get('pick', []):
                    # Combine all the info into a single record
                    full_record = {
                        **gameweek_info,
                        **entry_info,
                        **pick_data
                    }
                    all_picks.append(full_record)
        else:
            break # Stop if the next page doesn't exist

# 3. Create the DataFrame
fgo_df = pd.DataFrame(all_picks)

# Optional: Clean up the 'points' and 'total_points' columns
fgo_df['points'] = pd.to_numeric(fgo_df['points'])
fgo_df['total_points'] = fgo_df['total_points'].str.replace(' Points', '', regex=False).astype(float)

# --- Display the Result ---
print("✅ DataFrame created successfully!")
print(f"Total rows: {len(fgo_df)}")
print("\n--- Sample of the final DataFrame ---")

fgo_df

✅ DataFrame created successfully!
Total rows: 110088

--- Sample of the final DataFrame ---


,gameweek,contestant_no,prize_pool,manager,entry_num,total_points,prize,player_name,points,position,is_captin,is_vice
0,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Sanchez,2.0,,False,False
1,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Veltman,1.0,,False,False
2,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Wan-Bissaka,12.0,,False,False
3,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Estupiñan,7.0,,False,False
4,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Mitoma,5.0,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
110083,38,256,"R 9,500.00",MJ23,2,26.0,,B.Fernandes,0.0,,True,False
110084,38,256,"R 9,500.00",MJ23,2,26.0,,Bailey,0.0,,False,False
110085,38,256,"R 9,500.00",MJ23,2,26.0,,Haaland,3.0,,False,True
110086,38,256,"R 9,500.00",MJ23,2,26.0,,Darwin,1.0,,False,False


In [25]:
fpl_players_df = pd.read_csv('../with new features/data/vaastav/data/2023-24/players_raw.csv')
fgo_names = fgo_df['player_name'].unique().tolist()
fpl_web_names = fpl_players_df['web_name'].unique().tolist()

In [26]:
# confirm names in fgo match those in fpl_web_names
match_df =[]
for name in fgo_names:
    if name in fpl_web_names:
        match_df.append({'name': name, 'has_match': True})
    else:
        match_df.append({'name': name, 'has_match': False})
match_df = pd.DataFrame(match_df)

match_df[~match_df['has_match']]['name'].to_list()

['Vinicius', 'Mitooma', 'Bradely', 'N.Semendo', 'De Bryune', 'Isak.']

In [27]:
fgo_missing_names_dict = {
    'Vinicius': 'Vinícius' ,
    'Mitooma': 'Mitoma',
    'Bradely': 'Bradley',
    'N.Semendo': 'N.Semedo',
    'De Bryune': 'De Bruyne',
    'Isak.': 'Isak'
}

# Use .map() to look up corrected names and .fillna() to keep original names that weren't in the dict.
fgo_df['player_name'] = fgo_df['player_name'].map(fgo_missing_names_dict).fillna(fgo_df['player_name'])

# Map: web_name -> element_type
position_map = fpl_players_df.set_index('web_name')['element_type'].to_dict()

# Map: web_name -> id
id_map = fpl_players_df.set_index('web_name')['id'].to_dict()

# This looks up each corrected player_name in your new dictionaries.
fgo_df['position'] = fgo_df['player_name'].map(position_map)
fgo_df['fpl_id'] = fgo_df['player_name'].map(id_map)

fgo_df.loc[fgo_df['fpl_id'] == 862, 'fpl_id'] = 204

In [28]:
# 1. Calculate the number of managers for each gameweek
# This calculates the size of each group and divides by 11
manager_counts = fgo_df.groupby('gameweek')['gameweek'].transform('size') / 11
fgo_df['managers'] = manager_counts

# 2. Calculate the ownership counts for each player within each gameweek
ownership_counts = fgo_df.groupby(['gameweek', 'player_name'])['player_name'].transform('size')
fgo_df['owned_by'] = ownership_counts

# 3. Calculate the ownership percentage in one go
# We need to get the manager count for each group to divide by
# manager_counts_for_calc = fgo_df.groupby(['gameweek', 'player_name'])['managers'].first()
ownership_percentage = round((fgo_df['owned_by']  / fgo_df['managers']) * 100, 2)
fgo_df['ownership_percent'] = ownership_percentage

fgo_df['score'] = fgo_df.apply(lambda row: row['points']/2 if row['is_captin'] else row['points']/1.5 if row['is_vice'] else row['points'], axis=1)
fgo_df

,gameweek,contestant_no,prize_pool,manager,entry_num,total_points,prize,player_name,points,position,is_captin,is_vice,fpl_id,managers,owned_by,ownership_percent,score
0,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Sanchez,2.0,1,False,False,145,66.0,1,1.52,2.0
1,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Veltman,1.0,2,False,False,151,66.0,9,13.64,1.0
2,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Wan-Bissaka,12.0,2,False,False,401,66.0,7,10.61,12.0
3,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Estupiñan,7.0,2,False,False,131,66.0,32,48.48,7.0
4,1,175,"R 8,180.18",Radynster,1,81.5,"R 5,596.88",Mitoma,5.0,3,False,False,143,66.0,16,24.24,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110083,38,256,"R 9,500.00",MJ23,2,26.0,,B.Fernandes,0.0,3,True,False,373,256.0,33,12.89,0.0
110084,38,256,"R 9,500.00",MJ23,2,26.0,,Bailey,0.0,3,False,False,34,256.0,6,2.34,0.0
110085,38,256,"R 9,500.00",MJ23,2,26.0,,Haaland,3.0,4,False,True,355,256.0,177,69.14,2.0
110086,38,256,"R 9,500.00",MJ23,2,26.0,,Darwin,1.0,4,False,False,293,256.0,4,1.56,1.0


## Combine with FPL merged data


In [29]:
fpl_23_24 = pd.read_csv('../with new features/data/joint/23-24/merged_player_data.csv')
players_preds_36 = pd.read_csv('../with new features/models/preds/players_preds_36.csv')
players_preds_37 = pd.read_csv('../with new features/models/preds/players_preds_37.csv')
players_preds_38 = pd.read_csv('../with new features/models/preds/players_preds_38.csv')
# fpl_23_24[['event', 'fpl_id', 'ownership_percent']]

In [30]:
# # Select only the necessary columns
# ownership_lookup = fgo_df[['gameweek', 'fpl_id', 'player_name','ownership_percent', 'score']].copy()

# # Ensure 'gameweek' is numeric
# ownership_lookup['gameweek'] = pd.to_numeric(ownership_lookup['gameweek'])

# # Drop duplicate entries to have one row per player per gameweek
# ownership_lookup.drop_duplicates(subset=['gameweek', 'fpl_id'], inplace=True)
# ownership_lookup = ownership_lookup.rename(columns={'gameweek':'round'})

# ownership_lookup[(ownership_lookup['round'] == 37) & (ownership_lookup['fpl_id'] == 362)]

,round,fpl_id,player_name,ownership_percent,score
104605,37,362,Palmer,84.77,14.0


In [31]:
# missing_ids = set(set(ownership_lookup['fpl_id'].unique()) - set(fpl_23_24['fpl_id'].unique()))
# print(len(missing_ids))

# # Get the player names for missing ids
# missing_players = ownership_lookup[ownership_lookup['fpl_id'].isin(missing_ids)].drop_duplicates(subset=['fpl_id'])
# missing_players[['fpl_id', 'player_name']]

20


,fpl_id,player_name
6,505,Ndombele
33,498,Forster
627,323,Macey
704,299,Henderson
711,35,Buendia
714,443,Dennis
3408,207,Lukaku
5962,502,Lloris
30624,384,Heaton
30634,717,Stutter


In [ ]:


# Select only the necessary columns
ownership_lookup = fgo_df[['gameweek', 'fpl_id', 'player_name','ownership_percent', 'score']].copy()

# Ensure 'gameweek' is numeric
ownership_lookup['gameweek'] = pd.to_numeric(ownership_lookup['gameweek'])

# Drop duplicate entries to have one row per player per gameweek
ownership_lookup.drop_duplicates(subset=['gameweek', 'fpl_id'], inplace=True)
ownership_lookup = ownership_lookup.rename(columns={'gameweek':'round'})

# ownership_lookup[(ownership_lookup['round'] == 37) & (ownership_lookup['fpl_id'] == 362)]

# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
final_df = pd.merge(
    fpl_23_24,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='right'               # Use 'left' to keep all rows from merged_player_df
)


# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
preds_36 = pd.merge(
    players_preds_36,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

preds_37 = pd.merge(
    players_preds_37,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

preds_38 = pd.merge(
    players_preds_38,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='left'               # Use 'left' to keep all rows from merged_player_df
)

final_df['score_bps'] = final_df['score'] - final_df['bonus']
preds_36['score_bps'] = preds_36['score'] - preds_36['bonus']
preds_37['score_bps'] = preds_37['score'] - preds_37['bonus']
preds_38['score_bps'] = preds_38['score'] - preds_38['bonus']

final_df['ownership_percent'] = final_df['ownership_percent'].fillna(0)
preds_36['ownership_percent'] = preds_36['ownership_percent'].fillna(0)
preds_37['ownership_percent'] = preds_37['ownership_percent'].fillna(0)
preds_38['ownership_percent'] = preds_38['ownership_percent'].fillna(0)

final_df.dropna(subset=['fpl_name'], inplace=True)


In [33]:
def process_fpl_rolling_stats(df, windows=[1, 3, 5]):
    """
    Calculates lagged rolling averages for a wide range of FPL features while handling
    missing gameweeks via reindexing, then filters back to the original row set.
    Optimized to avoid DataFrame fragmentation.
    """
    # 0. Define the features to roll
    features_to_roll = [
        'clean_sheets', 'expected_assists', 'expected_goal_involvements',
        'expected_goals', 'expected_goals_conceded', 'goals_conceded',
        'goals_scored', 'ict_index', 'influence', 'creativity', 'threat',
        'minutes', 'own_goals', 'penalties_missed', 'penalties_saved',
        'red_cards', 'yellow_cards', 'saves', 'starts', 'team_a_score',
        'team_h_score',  'goals', 'shots', 'xG', 'xA', 'assists_x',
        'key_passes', 'npg', 'npxG',  'xGChain', 'xGBuildup', 'xP', 'score_bps'
    ]

    # 1. Keep track of the original keys to filter back later
    original_keys = df[['fpl_id', 'round']].copy()

    # 2. Prepare the full grid of players and gameweeks (1-38)
    all_gws = range(1, 39)
    players = df['fpl_id'].unique()

    full_index = pd.MultiIndex.from_product(
        [players, all_gws],
        names=['fpl_id', 'round']
    )

    # 3. Reindex to ensure every player has exactly 38 rows
    name_map = df[['fpl_id', 'player_name']].drop_duplicates().set_index('fpl_id')['player_name']

    new_df = (
        df.set_index(['fpl_id', 'round'])
        .reindex(full_index)
        .reset_index()
    )

    # 4. Fill name column and prepare "filled" columns for calculation
    new_df['player_name'] = new_df['fpl_id'].map(name_map)

    # Efficiently create the "filled" columns
    filled_data = new_df[features_to_roll].fillna(0)
    filled_data.columns = [f'{c}_filled' for c in filled_data.columns]
    new_df = pd.concat([new_df, filled_data], axis=1)

    # 5. Sort for rolling calculations
    new_df = new_df.sort_values(['fpl_id', 'round'])

    # 6. Calculate Rolling Averages efficiently
    # We store new columns in a list to concat at once, avoiding fragmentation
    rolling_cols = []
    grouped = new_df.groupby('fpl_id')

    for w in windows:
        for feat in features_to_roll:
            col_name = f'{feat}_{w}'
            rolling_series = (
                grouped[f'{feat}_filled']
                .transform(lambda x: x.rolling(window=w, min_periods=1).mean().shift(1))
                .round(2)
            )
            rolling_series.name = col_name
            rolling_cols.append(rolling_series)

    # Join all rolling columns at once
    new_df = pd.concat([new_df] + rolling_cols, axis=1)

    # 7. Filter back to only include rows that were in the original final_df
    result_df = pd.merge(original_keys, new_df, on=['fpl_id', 'round'], how='left')

    # Clean up temporary columns
    filled_cols = [c for c in result_df.columns if c.endswith('_filled')]
    result_df = result_df.drop(columns=filled_cols)

    # De-fragment the final frame
    return result_df.copy()

# Execute the processing
processed_df = process_fpl_rolling_stats(final_df)

# Inspection: Check Haaland's form trend across multiple features
processed_df.query("player_name == 'Haaland'")[
    ['round', 'minutes', 'minutes_3', 'score_bps', 'score_bps_3', 'xG_3', 'xA_3', 'xP_3']
]


,round,minutes,minutes_3,minutes_3,score_bps,score_bps_3,xG_3,xG_3,xA_3,xA_3,xP_3,xP_3
8,1,79.0,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
139,2,90.0,26.33,79.00,2.0,10.00,0.29,0.88,0.07,0.22,1.83,5.50
257,3,90.0,56.33,84.50,4.0,6.00,0.43,0.65,0.11,0.17,4.50,6.75
379,4,90.0,86.33,86.33,17.0,5.33,1.13,1.13,0.14,0.14,7.10,7.10
497,5,90.0,90.00,90.00,6.0,7.67,1.28,1.28,0.29,0.29,8.87,8.87
620,6,90.0,90.00,90.00,6.0,9.00,2.12,2.12,0.26,0.26,9.87,9.87
731,7,90.0,90.00,90.00,2.0,9.67,1.73,1.73,0.24,0.24,10.60,10.60
889,8,90.0,90.00,90.00,2.0,4.67,1.31,1.31,0.03,0.03,8.90,8.90
992,9,90.0,90.00,90.00,6.0,3.33,0.34,0.34,0.18,0.18,6.50,6.50
1100,10,90.0,90.00,90.00,13.0,3.33,0.09,0.09,0.18,0.18,4.83,4.83


In [34]:
processed_df.to_csv('./fantasyGo_FPL.csv')
preds_36.to_csv('./fantasyGo_preds_36.csv')
preds_37.to_csv('./fantasyGo_preds_37.csv')
preds_38.to_csv('./fantasyGo_preds_38.csv')

# 24/25


In [35]:
fgo_24_25 = pd.read_csv('./fantasyGo 24-25.csv')
fgo_24_25 #[['Gameweek', 'Username', 'Points', 'Players']]

,userContestId,Gameweek,Event,Season,CompletedAt,ContestEntries,EntryFees,Rank,Username,Points,Formation,Players
0,90371,Gameweek 38,English Premier League,2024/25,2025-05-26T06:46:58.502Z,1785,10,1,big mouth,94.5,1-3-4-3,"[{'name': 'Gakpo', 'firstName': 'Cody', 'lastN..."
1,94896,Gameweek 38,English Premier League,2024/25,2025-05-26T06:46:58.502Z,1785,10,2,undefeateddevil,92.0,1-5-3-2,"[{'name': 'M.Salah', 'firstName': 'Mohamed', '..."
2,96685,Gameweek 38,English Premier League,2024/25,2025-05-26T06:46:58.502Z,1785,10,3,putthemoneyonthetable,90.5,1-3-5-2,"[{'name': 'M.Salah', 'firstName': 'Mohamed', '..."
3,94301,Gameweek 38,English Premier League,2024/25,2025-05-26T06:46:58.502Z,1785,10,4,EDWIN K,90.0,1-3-5-2,"[{'name': 'M.Salah', 'firstName': 'Mohamed', '..."
4,92438,Gameweek 38,English Premier League,2024/25,2025-05-26T06:46:58.502Z,1785,10,4,user92057197,90.0,1-3-5-2,"[{'name': 'Gakpo', 'firstName': 'Cody', 'lastN..."
...,...,...,...,...,...,...,...,...,...,...,...,...
4370,43377,Gameweek 34,English Premier League,2024/25,2025-05-02T07:14:00.895Z,1474,10,1470,Samdran,21.0,1-4-4-2,"[{'name': 'Unknown (6802)', 'firstName': 'Unkn..."
4371,38703,Gameweek 34,English Premier League,2024/25,2025-05-02T07:14:00.895Z,1474,10,1471,Shhhhh,20.5,1-5-4-1,"[{'name': 'Unknown (6815)', 'firstName': 'Unkn..."
4372,44162,Gameweek 34,English Premier League,2024/25,2025-05-02T07:14:00.895Z,1474,10,1472,qalabocha,17.0,1-3-5-2,"[{'name': 'Unknown (6958)', 'firstName': 'Unkn..."
4373,45221,Gameweek 34,English Premier League,2024/25,2025-05-02T07:14:00.895Z,1474,10,1473,Sihlewest,13.0,1-4-3-3,"[{'name': 'Unknown (6809)', 'firstName': 'Unkn..."


In [120]:
def extract_player_details(row):
    try:
        # Convert string representation of list to actual list
        players_list = ast.literal_eval(row['Players'])

        # Create a list of dictionaries for each player in this entry
        return [{
            'gameweek': row['Gameweek'].split(' ')[1],
            'contestant_no': row['ContestEntries'],
            'manager': row['Username'],
            'total_points': row['Points'],
            'player_name': p.get('name'),
            'full_name': f"{p.get('firstName', '')} {p.get('lastName', '')}".strip(),
            'points': p.get('pointsInThisContest'),
            'position': p.get('position'),
            'is_captin': p.get('isCaptain', False),
            'is_vice': p.get('isViceCaptain', False),
            'season': row['Season']
        } for p in players_list]
    except (ValueError, SyntaxError):
        return []

# 2. Extract data into a list of all player-gameweek records
all_records = []
for _, row in fgo_24_25.iterrows():
    all_records.extend(extract_player_details(row))

fgo_24_25_df = pd.DataFrame(all_records)
fgo_24_25_df

,gameweek,contestant_no,manager,total_points,player_name,full_name,points,position,is_captin,is_vice,season
0,38,1785,big mouth,94.5,Gakpo,Cody Gakpo,10.5,forward,False,True,2024/25
1,38,1785,big mouth,94.5,M.Salah,Mohamed Salah,10.0,midfielder,False,False,2024/25
2,38,1785,big mouth,94.5,Cunha,Matheus Santos Carneiro Da Cunha,2.0,forward,False,False,2024/25
3,38,1785,big mouth,94.5,Kerkez,Milos Kerkez,6.0,defender,False,False,2024/25
4,38,1785,big mouth,94.5,Semenyo,Antoine Semenyo,16.0,midfielder,False,False,2024/25
...,...,...,...,...,...,...,...,...,...,...,...
48079,34,1474,Shumza506,10.0,Unknown (6818),Unknown (6818) Unknown (6818),2.0,None,False,False,2024/25
48080,34,1474,Shumza506,10.0,Unknown (6947),Unknown (6947) Unknown (6947),3.0,None,False,True,2024/25
48081,34,1474,Shumza506,10.0,Unknown (7339),Unknown (7339) Unknown (7339),1.0,None,False,False,2024/25
48082,34,1474,Shumza506,10.0,Unknown (7397),Unknown (7397) Unknown (7397),4.0,None,False,False,2024/25


### FGO + FPL


In [123]:
fpl_24_25 = pd.read_csv('../with new features/data/vaastav/data/2024-25/players_raw.csv')

fpl_24_25.loc[(fpl_24_25['first_name'] == 'Luke') & (fpl_24_25['second_name'] == 'Thomas'), 'web_name'] = "Luke Thomas"
fpl_24_25.loc[(fpl_24_25['first_name'] == 'Thomas') & (fpl_24_25['second_name'] == 'Partey'), 'web_name'] = "Thomas Partey"
fpl_24_25[(fpl_24_25['web_name'] == 'Luke Thomas') | (fpl_24_25['web_name'] == 'Thomas Partey')]

fgo_24_25_names = fgo_24_25_df['player_name'].unique().tolist()
fpl_web_names = fpl_24_25['web_name'].unique().tolist()

In [124]:
fgo_24_25_names = fgo_24_25_df['player_name'].unique().tolist()
fpl_web_names = fpl_24_25['web_name'].unique().tolist()

# confirm names in fgo match those in fpl_web_names
match_df =[]
for name in fgo_24_25_names:
    if name in fpl_web_names:
        match_df.append({'name': name, 'has_match': True})
    else:
        match_df.append({'name': name, 'has_match': False})
match_df = pd.DataFrame(match_df)

match_df[~match_df['has_match']]['name'].to_list()

['Partey',
 'Thomas',
 'Unknown (6811)',
 'Unknown (6812)',
 'Unknown (6816)',
 'Unknown (6915)',
 'Unknown (6947)',
 'Unknown (6958)',
 'Unknown (7041)',
 'Unknown (7150)',
 'Unknown (7301)',
 'Unknown (7332)',
 'Unknown (7430)',
 'Unknown (6815)',
 'Unknown (6933)',
 'Unknown (6937)',
 'Unknown (6999)',
 'Unknown (7124)',
 'Unknown (7295)',
 'Unknown (7322)',
 'Unknown (7326)',
 'Unknown (7419)',
 'Unknown (7464)',
 'Unknown (7470)',
 'Unknown (6813)',
 'Unknown (6814)',
 'Unknown (6831)',
 'Unknown (6955)',
 'Unknown (7102)',
 'Unknown (7197)',
 'Unknown (7247)',
 'Unknown (7270)',
 'Unknown (7287)',
 'Unknown (7339)',
 'Unknown (7415)',
 'Unknown (7304)',
 'Unknown (7380)',
 'Unknown (6804)',
 'Unknown (7022)',
 'Unknown (7279)',
 'Unknown (7398)',
 'Unknown (6973)',
 'Unknown (6989)',
 'Unknown (7027)',
 'Unknown (7036)',
 'Unknown (7085)',
 'Unknown (7099)',
 'Unknown (7100)',
 'Unknown (7299)',
 'Unknown (6821)',
 'Unknown (7021)',
 'Unknown (7128)',
 'Unknown (7132)',
 'Unknown

In [ ]:

fgo_missing_names_dict = {
    'Partey': 'Thomas Partey',
    'Thomas': 'Luke Thomas'
}

# Use .map() to look up corrected names and .fillna() to keep original names that weren't in the dict.
fgo_24_25_df['player_name'] = fgo_24_25_df['player_name'].map(fgo_missing_names_dict).fillna(fgo_24_25_df['player_name'])

# Map: web_name -> element_type
position_map = fpl_24_25.set_index('web_name')['element_type'].to_dict()

# Map: web_name -> id
id_map = fpl_24_25.set_index('web_name')['id'].to_dict()

# This looks up each corrected player_name in your new dictionaries.
fgo_24_25_df['position'] = fgo_24_25_df['player_name'].map(position_map)
fgo_24_25_df['fpl_id'] = fgo_24_25_df['player_name'].map(id_map)

# fgo_24_25_df.loc[fgo_24_25_df['fpl_id'] == 862, 'fpl_id'] = 204

# Drop fields with missing fpl_id
fgo_24_25_df = fgo_24_25_df.dropna(subset=['fpl_id'])

In [134]:
# 1. Calculate the number of managers for each gameweek
# This calculates the size of each group and divides by 11
manager_counts = fgo_24_25_df.groupby('gameweek')['gameweek'].transform('size') / 11
fgo_24_25_df['managers'] = manager_counts

# 2. Calculate the ownership counts for each player within each gameweek
ownership_counts = fgo_24_25_df.groupby(['gameweek', 'player_name'])['player_name'].transform('size')
fgo_24_25_df['owned_by'] = ownership_counts

# 3. Calculate the ownership percentage in one go
# We need to get the manager count for each group to divide by
# manager_counts_for_calc = fgo_24_25_df.groupby(['gameweek', 'player_name'])['managers'].first()
ownership_percentage = round((fgo_24_25_df['owned_by']  / fgo_24_25_df['managers']) * 100, 2)
fgo_24_25_df['ownership_percent'] = ownership_percentage

fgo_24_25_df['score'] = fgo_24_25_df.apply(lambda row: row['points']/2 if row['is_captin'] else row['points']/1.5 if row['is_vice'] else row['points'], axis=1)
fgo_24_25_df

,gameweek,contestant_no,manager,total_points,player_name,full_name,points,position,is_captin,is_vice,season,fpl_id,managers,owned_by,ownership_percent,score
0,38,1785,big mouth,94.5,Gakpo,Cody Gakpo,10.5,4.0,False,True,2024/25,321.0,1785.0,118,6.61,7.0
1,38,1785,big mouth,94.5,M.Salah,Mohamed Salah,10.0,3.0,False,False,2024/25,328.0,1785.0,457,25.60,10.0
2,38,1785,big mouth,94.5,Cunha,Matheus Santos Carneiro Da Cunha,2.0,4.0,False,False,2024/25,541.0,1785.0,135,7.56,2.0
3,38,1785,big mouth,94.5,Kerkez,Milos Kerkez,6.0,2.0,False,False,2024/25,70.0,1785.0,447,25.04,6.0
4,38,1785,big mouth,94.5,Semenyo,Antoine Semenyo,16.0,3.0,False,False,2024/25,78.0,1785.0,259,14.51,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45436,36,1372,shannie,42.0,Haaland,Erling Haaland,4.0,4.0,True,False,2024/25,351.0,672.0,186,27.68,2.0
45437,36,1372,shannie,42.0,Amad,Amad Diallo,2.0,3.0,False,False,2024/25,364.0,672.0,13,1.93,2.0
45438,36,1372,shannie,42.0,Aina,Ola Aina,0.0,2.0,False,False,2024/25,422.0,672.0,38,5.65,0.0
45439,36,1372,shannie,42.0,N.Williams,Neco Williams,1.0,2.0,False,False,2024/25,437.0,672.0,117,17.41,1.0


In [135]:
fpl_merged = pd.read_csv('../with new features/data/joint/23-24/merged_player_data.csv')



# Select only the necessary columns
ownership_lookup = fgo_24_25_df[['gameweek', 'fpl_id', 'player_name','ownership_percent', 'score']].copy()

# Ensure 'gameweek' is numeric
ownership_lookup['gameweek'] = pd.to_numeric(ownership_lookup['gameweek'])

# Drop duplicate entries to have one row per player per gameweek
ownership_lookup.drop_duplicates(subset=['gameweek', 'fpl_id'], inplace=True)
ownership_lookup = ownership_lookup.rename(columns={'gameweek':'round'})

# ownership_lookup[(ownership_lookup['round'] == 37) & (ownership_lookup['fpl_id'] == 362)]

# Merge on both 'gameweek' and 'fpl_id' to ensure correct matching
final_24_25_df = pd.merge(
    fpl_merged,
    ownership_lookup,
    on=['round', 'fpl_id'],  # Use a list for multiple merge keys
    how='right'               # Use 'left' to keep all rows from merged_player_df
)


final_24_25_df['score_bps'] = final_24_25_df['score'] - final_24_25_df['bonus']
final_24_25_df['ownership_percent'] = final_24_25_df['ownership_percent'].fillna(0)
final_24_25_df.dropna(subset=['fpl_name'], inplace=True)


In [ ]:
def process_fpl_rolling_stats(df, windows=[1, 3, 5]):
    """
    Calculates lagged rolling averages for a wide range of FPL features while handling
    missing gameweeks via reindexing, then filters back to the original row set.
    Optimized to avoid DataFrame fragmentation.
    """
    # 0. Define the features to roll
    features_to_roll = [
        'clean_sheets', 'expected_assists', 'expected_goal_involvements',
        'expected_goals', 'expected_goals_conceded', 'goals_conceded',
        'goals_scored', 'ict_index', 'influence', 'creativity', 'threat',
        'minutes', 'own_goals', 'penalties_missed', 'penalties_saved',
        'red_cards', 'yellow_cards', 'saves', 'starts', 'team_a_score',
        'team_h_score',  'goals', 'shots', 'xG', 'xA', 'assists_x',
        'key_passes', 'npg', 'npxG',  'xGChain', 'xGBuildup', 'xP', 'score_bps'
    ]

    # 1. Keep track of the original keys to filter back later
    original_keys = df[['fpl_id', 'round']].copy()

    # 2. Prepare the full grid of players and gameweeks (1-38)
    all_gws = range(1, 39)
    players = df['fpl_id'].unique()

    full_index = pd.MultiIndex.from_product(
        [players, all_gws],
        names=['fpl_id', 'round']
    )

    # 3. Reindex to ensure every player has exactly 38 rows
    name_map = df[['fpl_id', 'player_name']].drop_duplicates().set_index('fpl_id')['player_name']

    new_df = (
        df.set_index(['fpl_id', 'round'])
        .reindex(full_index)
        .reset_index()
    )

    # 4. Fill name column and prepare "filled" columns for calculation
    new_df['player_name'] = new_df['fpl_id'].map(name_map)

    # Efficiently create the "filled" columns
    filled_data = new_df[features_to_roll].fillna(0)
    filled_data.columns = [f'{c}_filled' for c in filled_data.columns]
    new_df = pd.concat([new_df, filled_data], axis=1)

    # 5. Sort for rolling calculations
    new_df = new_df.sort_values(['fpl_id', 'round'])

    # 6. Calculate Rolling Averages efficiently
    # We store new columns in a list to concat at once, avoiding fragmentation
    rolling_cols = []
    grouped = new_df.groupby('fpl_id')

    for w in windows:
        for feat in features_to_roll:
            col_name = f'{feat}_{w}'
            rolling_series = (
                grouped[f'{feat}_filled']
                .transform(lambda x: x.rolling(window=w, min_periods=1).mean().shift(1))
                .round(2)
            )
            rolling_series.name = col_name
            rolling_cols.append(rolling_series)

    # Join all rolling columns at once
    new_df = pd.concat([new_df] + rolling_cols, axis=1)

    # 7. Filter back to only include rows that were in the original final_24_25_df
    result_df = pd.merge(original_keys, new_df, on=['fpl_id', 'round'], how='left')

    # Clean up temporary columns
    filled_cols = [c for c in result_df.columns if c.endswith('_filled')]
    result_df = result_df.drop(columns=filled_cols)

    # De-fragment the final frame
    return result_df.copy()

# Execute the processing
processed_24_25_df = process_fpl_rolling_stats(final_24_25_df)

processed_24_25_df.to_csv('./fantasyGo_FPL_24_25.csv')



,round,minutes,minutes_3,minutes_3,score_bps,score_bps_3,xG_3,xG_3,xA_3,xA_3,xP_3,xP_3
21,38,6.0,63.67,3.67,6.0,0.67,0.01,0.0,0.01,0.0,1.43,0.27
143,37,11.0,88.67,0.00,2.0,0.00,0.05,0.0,0.01,0.0,1.67,0.00


In [139]:

# Inspection: Check Haaland's form trend across multiple features
processed_24_25_df.query("player_name == 'Cole'")[
    ['round', 'minutes', 'minutes_3', 'score_bps', 'score_bps_3', 'xG_3', 'xA_3', 'xP_3']
]

,round,minutes,minutes_3,minutes_3,score_bps,score_bps_3,xG_3,xG_3,xA_3,xA_3,xP_3,xP_3
